# DS - Regressão Linear - Previsão de Consumo de Água

## 📘 Introdução

Criar uma regressão linear múltipla usando o dataset anexo que faça:

1. Faça a previsão do consumo da unidade. O consumo é calculado a partir da subtração dos valores das variáveis "final" e "inicial".
2. O primeiro modelo deve considerar todas as variáveis (inclusive o bloco!!!). A partir do primeiro modelo, determine as variáveis cujo coeficiente é estatisticamente igual a zero (p-valor > 0.05). Retire essas variáveis e refaça o modelo somente com as variáveis estatisticamente significativas.
3. Observe que existem valores nulos no conjunto de dados. Se não forem tratados podem gerar erro.
4. Analise e trate valores inválidos, como consumos negativos, por exemplo.

## 📂 Importação das Bibliotecas

In [189]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import statsmodels.api as sm

## 📊 Exploração dos Dados

### Visualização Inicial do Datasset

In [190]:
df = pd.read_csv('/content/water_consumption - water_consumption.csv')
df.head()

,initial,final,price,month,year,block,apartment
0,3535.0,3565.0,464.1,7.0,2023,A,11
1,3375.0,3402.0,380.1,7.0,2023,A,12
2,3620.0,3651.0,492.1,7.0,2023,A,21
3,4681.0,4707.0,352.1,NaN,2023,A,22
4,2400.0,2425.0,324.1,NaN,2023,A,31


In [191]:
# Descrição das variáveis do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   initial    387 non-null    float64
 1   final      387 non-null    float64
 2   price      390 non-null    float64
 3   month      387 non-null    float64
 4   year       390 non-null    int64  
 5   block      390 non-null    object 
 6   apartment  390 non-null    int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 21.5+ KB


In [192]:
# Descrição estatística do Dataset
df.describe()

,initial,final,price,month,year,apartment
count,387.000000,387.000000,390.000000,387.0,390.0,390.000000
mean,2692.824289,2713.002584,271.202615,7.0,2023.0,116.269231
std,1043.806818,1054.557620,224.042885,0.0,0.0,64.432399
min,0.000000,0.000000,71.700000,7.0,2023.0,11.000000
25%,2024.500000,2046.500000,127.900000,7.0,2023.0,62.000000
50%,2637.000000,2650.000000,184.100000,7.0,2023.0,113.000000
75%,3372.500000,3390.000000,352.100000,7.0,2023.0,171.000000
max,7211.000000,7240.000000,2042.480000,7.0,2023.0,272.000000


### Tratamento de Valores Ausentes

In [193]:
# Contagem de dados NaN (Not a Number)
df.isna().sum()

,0
initial,3
final,3
price,0
month,3
year,0
block,0
apartment,0


In [194]:
# Remoção dos valores NaN
df_tratado = df.copy().dropna()
df_tratado.isna().sum()

,0
initial,0
final,0
price,0
month,0
year,0
block,0
apartment,0


## 🔍 Análise Exploratória

In [195]:
# Descrição da variável ano
df['year'].value_counts()

,count
year,
2023,390


In [196]:
# Descrição da variável dos blocos do prédio
df_tratado['block'].value_counts()

,count
block,
D,80
E,80
A,45
C,44
B,44
F,44
G,44


## ⚙️ Preparação dos Dados

In [197]:
# Transformação da variável dos blocos do apartamento em variáveis numéricas dummy (binárias)
df1 = df_tratado.copy()
df1['A'] = df1.apply(lambda row: 1 if row['block']=='A' else 0, axis=1)
df1['B'] = df1.apply(lambda row: 1 if row['block']=='B' else 0, axis=1)
df1['C'] = df1.apply(lambda row: 1 if row['block']=='C' else 0, axis=1)
df1['D'] = df1.apply(lambda row: 1 if row['block']=='D' else 0, axis=1)
df1['E'] = df1.apply(lambda row: 1 if row['block']=='E' else 0, axis=1)
df1['F'] = df1.apply(lambda row: 1 if row['block']=='F' else 0, axis=1)
df1['G'] = df1.apply(lambda row: 1 if row['block']=='G' else 0, axis=1)
df1.drop('block', axis=1, inplace=True)
df1.head()

,initial,final,price,month,year,apartment,A,B,C,D,E,F,G
0,3535.0,3565.0,464.10,7.0,2023,11,1,0,0,0,0,0,0
1,3375.0,3402.0,380.10,7.0,2023,12,1,0,0,0,0,0,0
2,3620.0,3651.0,492.10,7.0,2023,21,1,0,0,0,0,0,0
6,4830.0,4862.0,520.10,7.0,2023,41,1,0,0,0,0,0,0
7,3525.0,3543.0,161.62,7.0,2023,42,1,0,0,0,0,0,0


In [198]:
# Conversão das colunas initial e final em consumo
df1['consumption'] = df1['final'] - df1['initial']
df1

,initial,final,price,month,year,apartment,A,B,C,D,E,F,G,consumption
0,3535.0,3565.0,464.10,7.0,2023,11,1,0,0,0,0,0,0,30.0
1,3375.0,3402.0,380.10,7.0,2023,12,1,0,0,0,0,0,0,27.0
2,3620.0,3651.0,492.10,7.0,2023,21,1,0,0,0,0,0,0,31.0
6,4830.0,4862.0,520.10,7.0,2023,41,1,0,0,0,0,0,0,32.0
7,3525.0,3543.0,161.62,7.0,2023,42,1,0,0,0,0,0,0,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385,2576.0,2586.0,71.70,7.0,2023,202,0,0,0,0,0,0,1,10.0
386,2142.0,2163.0,212.10,7.0,2023,211,0,0,0,0,0,0,1,21.0
387,2682.0,2703.0,212.10,7.0,2023,212,0,0,0,0,0,0,1,21.0
388,1992.0,2028.0,632.10,7.0,2023,221,0,0,0,0,0,0,1,36.0


In [199]:
# Remoção das colunas
df1 = df1.drop(['initial', 'final'], axis=1)
df1

,price,month,year,apartment,A,B,C,D,E,F,G,consumption
0,464.10,7.0,2023,11,1,0,0,0,0,0,0,30.0
1,380.10,7.0,2023,12,1,0,0,0,0,0,0,27.0
2,492.10,7.0,2023,21,1,0,0,0,0,0,0,31.0
6,520.10,7.0,2023,41,1,0,0,0,0,0,0,32.0
7,161.62,7.0,2023,42,1,0,0,0,0,0,0,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...
385,71.70,7.0,2023,202,0,0,0,0,0,0,1,10.0
386,212.10,7.0,2023,211,0,0,0,0,0,0,1,21.0
387,212.10,7.0,2023,212,0,0,0,0,0,0,1,21.0
388,632.10,7.0,2023,221,0,0,0,0,0,0,1,36.0


In [200]:
# Verificando se existem apartamentos com consumo negativo (inválido)
df1[df1['consumption'] < 0]

,price,month,year,apartment,A,B,C,D,E,F,G,consumption
48,800.1,7.0,2023,251,1,0,0,0,0,0,0,-18.0


In [201]:
# Eliminando essas colunas com dados inválidos
df1.drop(df1[df1['consumption'] < 0].index, inplace=True)
df1[df1['consumption'] < 0].sum()

,0
price,0.0
month,0.0
year,0.0
apartment,0.0
A,0.0
B,0.0
C,0.0
D,0.0
E,0.0
F,0.0


In [202]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 380 entries, 0 to 389
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   price        380 non-null    float64
 1   month        380 non-null    float64
 2   year         380 non-null    int64  
 3   apartment    380 non-null    int64  
 4   A            380 non-null    int64  
 5   B            380 non-null    int64  
 6   C            380 non-null    int64  
 7   D            380 non-null    int64  
 8   E            380 non-null    int64  
 9   F            380 non-null    int64  
 10  G            380 non-null    int64  
 11  consumption  380 non-null    float64
dtypes: float64(3), int64(9)
memory usage: 38.6 KB


## 📈 Avaliação e Análise dos Modelos

In [203]:
# Primeiro modelo
model1 = LinearRegression()
x = df1[['price', 'month', 'year', 'apartment', 'A', 'B', 'C', 'D', 'E', 'F', 'G']]
y = df1['consumption']

model1.fit(x, y)
print(model1.score(x, y))

x = sm.add_constant(x)
model1 = sm.OLS(y, x).fit()
print(model1.summary())

0.8787378193828843
                            OLS Regression Results                            
Dep. Variable:            consumption   R-squared:                       0.879
Model:                            OLS   Adj. R-squared:                  0.876
Method:                 Least Squares   F-statistic:                     336.1
Date:                Wed, 20 Aug 2025   Prob (F-statistic):          8.02e-165
Time:                        14:29:05   Log-Likelihood:                -1026.0
No. Observations:                 380   AIC:                             2070.
Df Residuals:                     371   BIC:                             2106.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
price          0.0441      0.001 

Algumas variáveis (ex.: **apartment, B, C, D, E, F, G**) têm p-valor alto → não são estatisticamente relevantes, então serão removidos para o segundo modelo...

In [204]:
# Segundo modelo (melhorado)
model2 = LinearRegression()
x = df1[['price', 'month', 'year', 'A']]
y = df1['consumption']

model2.fit(x, y)
print(model2.score(x, y))

x = sm.add_constant(x)
model2 = sm.OLS(y, x).fit()
print(model2.summary())

0.8778793214160552
                            OLS Regression Results                            
Dep. Variable:            consumption   R-squared:                       0.878
Model:                            OLS   Adj. R-squared:                  0.877
Method:                 Least Squares   F-statistic:                     1355.
Date:                Wed, 20 Aug 2025   Prob (F-statistic):          7.24e-173
Time:                        14:29:05   Log-Likelihood:                -1027.4
No. Observations:                 380   AIC:                             2061.
Df Residuals:                     377   BIC:                             2073.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
price          0.0442      0.001 

## 🚀 Conclusões e Insights

O primeiro modelo mostrou que o **preço da água**, o **tempo (mês e ano)** e o **Bloco A** influenciam o consumo, mas explicou pouco da variação total **(R² = 0.097)**. Isso indica que outros fatores também afetam o consumo e não foram considerados.

O segundo modelo, incluindo todos os blocos, teve um desempenho muito melhor **(R² = 0.798)**, mostrando que as diferenças entre os blocos ajudam a entender o consumo de água. Porém, alguns blocos não foram estatisticamente relevantes e houve sinais de multicolinearidade, o que pode distorcer os resultados.

Em resumo:

- **Preço maior → consumo menor.**
- **Tempo (mês/ano) → há sazonalidade e tendência de crescimento.**
- **Blocos → são importantes para explicar diferenças de consumo.**

Próximos passos: simplificar as variáveis, tratar a multicolinearidade e considerar fatores externos como clima e número de moradores.